# 01. Unstructured, Structured Pruning, ShortGPT, and the Lottery Ticket Hypothesis

- **Category:** 15_compression_peft_and_speculative_inference / 01_pruning_sparsity_lowrank_and_distillation
- **Lineage & Citations:** Han et al. (2015, *Learning both Weights and Connections for Efficient Neural Networks*, NeurIPS); Frankle & Carbin (2018, *The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks*, ICLR); Men et al. (2024, *ShortGPT: Layers in Large Language Models are More Redundant Than You Think*, arXiv:2403.03853).
- **Purpose:** Implement and verify foundational model compression across four paradigms: scalar unstructured magnitude thresholding, dense structured channel/row $\ell_2$ norm slicing, zero-shot transformer layer pruning via ShortGPT cosine Block Influence (BI), and winning subnetwork extraction via the Lottery Ticket Hypothesis.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [14. Quantisation / 06. Quantization Method Selection Matrix and Head to Head Benchmark](../../../notebooks/14_quantisation/05_kv_attention_optim_and_benchmarks/06_quantization_method_selection_matrix_and_head_to_head_benchmark.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. 2:4 Semi-Structured Sparsity and RigL Dynamic Sparse](02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT deploy unstructured sparsity on standard dense GPU GEMM pipelines without sparse hardware support (unstructured zeros incur indexing overhead and slow down inference); do NOT prune transformer layers using ShortGPT without validating calibration Block Influence across diverse domains (pruning active layers with $\text{BI} > 0.05$ causes severe semantic collapse).

## Mathematical Foundations & Governing Equations

1. **Unstructured Magnitude Thresholding**:
   $$\tau = \text{Quantile}_{s}(|W|), \quad M_{ij} = \mathbb{I}(|W_{ij}| \ge \tau), \quad W^{\text{unstr}} = W \odot M$$
2. **Structured Row / Channel $\ell_2$ Norm Pruning**:
   $$s_i = \|W_{i, :}\|_2 = \sqrt{\sum_{j=1}^n W_{ij}^2}, \quad \mathcal{I} = \text{Top-}k(s, k = \lfloor \rho \cdot m \rfloor), \quad W^{\text{struct}} = W[\mathcal{I}, :] \in \mathbb{R}^{k \times n}$$
3. **ShortGPT Cosine Block Influence (BI)**:
   $$\text{BI}_l = 1 - \frac{1}{B} \sum_{b=1}^B \frac{X_{l, b} \cdot X_{l+1, b}}{\|X_{l, b}\|_2 \|X_{l+1, b}\|_2 + \epsilon}$$
4. **Lottery Ticket Subnetwork Extraction**:
   $$M = \mathbb{I}(|W_T| \ge \tau_s), \quad W_{\text{ticket}} = W_0 \odot M$$

### Asymptotic Complexity
- Unstructured Pruning: $\mathcal{O}(mn \log(mn))$ sorting, $\mathcal{O}(mn)$ memory.
- Structured Pruning: $\mathcal{O}(mn + m \log m)$ row norm & sort, $\mathcal{O}(kn)$ sliced dense memory.
- ShortGPT Block Influence: $\mathcal{O}(B \cdot d)$ per layer, zero additional memory.

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/01_unstructured_structured_shortgpt_and_lottery_ticket.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} golden tensors from {WEIGHTS_PATH.name}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def unstructured_magnitude_prune(w: torch.Tensor, sparsity: float = 0.5):
    k = max(1, int(sparsity * w.numel()))
    tau = torch.kthvalue(torch.abs(w).flatten(), k).values
    mask = (torch.abs(w) >= tau).to(torch.float32)
    return w * mask, mask

def structured_row_prune(w: torch.Tensor, keep_ratio: float = 0.75):
    row_norms = torch.norm(w, p=2, dim=1)
    k_rows = max(1, int(keep_ratio * w.shape[0]))
    indices = torch.topk(row_norms, k_rows, largest=True).indices.sort().values
    return w[indices, :], indices.to(torch.float32)

def compute_block_influence(x_in: torch.Tensor, x_out: torch.Tensor) -> float:
    dot = torch.sum(x_in * x_out, dim=-1)
    norm_in = torch.norm(x_in, p=2, dim=-1)
    norm_out = torch.norm(x_out, p=2, dim=-1)
    cos = dot / (norm_in * norm_out + 1e-8)
    return float(1.0 - torch.mean(cos))

def lottery_ticket_rewind(w_initial: torch.Tensor, w_trained: torch.Tensor, sparsity: float = 0.5):
    k = max(1, int(sparsity * w_trained.numel()))
    tau = torch.kthvalue(torch.abs(w_trained).flatten(), k).values
    mask = (torch.abs(w_trained) >= tau).to(torch.float32)
    return w_initial * mask, mask

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-6

# 2. Unstructured magnitude pruning parity
w_unstr, mask_unstr = unstructured_magnitude_prune(weights["w"], sparsity=0.5)
assert float(torch.max(torch.abs(w_unstr - weights["w_unstructured"]))) < 1e-6
assert float(torch.max(torch.abs(mask_unstr - weights["mask_unstructured"]))) < 1e-6

# 3. Structured row pruning parity
w_str, indices_str = structured_row_prune(weights["w"], keep_ratio=0.75)
assert float(torch.max(torch.abs(w_str - weights["w_structured"]))) < 1e-6
assert float(torch.max(torch.abs(indices_str - weights["indices_structured"]))) < 1e-6

# 4. ShortGPT Block Influence parity
bi_red = compute_block_influence(weights["x_in"], weights["x_out_red"])
bi_act = compute_block_influence(weights["x_in"], weights["x_out_act"])
bi_vec = torch.tensor([bi_red, bi_act], dtype=torch.float32)
assert float(torch.max(torch.abs(bi_vec - weights["bi_scores"]))) < 1e-6

# 5. Winning lottery ticket subnetwork parity
w_sub, mask_lottery = lottery_ticket_rewind(weights["w"], weights["w_trained"], sparsity=0.5)
assert float(torch.max(torch.abs(w_sub - weights["w_lottery_subnetwork"]))) < 1e-6
assert float(torch.max(torch.abs(mask_lottery - weights["mask_lottery"]))) < 1e-6
print("All 5 pruning implementations verified against seed-42 golden tensors!")

In [ ]:
# Empirical evaluation: Sparsity, Compression & Block Influence
sparsity_unstr = float(torch.mean((w_unstr == 0.0).float()))
row_reduction = 1.0 - (w_str.shape[0] / weights["w"].shape[0])
print(f"[1] Unstructured Sparsity: {sparsity_unstr * 100:.1f}%")
print(f"[2] Structured Row Reduction: {row_reduction * 100:.1f}% ({weights['w'].shape} -> {w_str.shape})")
print(f"[3] ShortGPT BI Redundant Block: {bi_red:.6f} (Pruning Candidate!)")
print(f"[4] ShortGPT BI Active Block:    {bi_act:.6f} (Preserve!)")
print(f"[5] Lottery Ticket Active Mask Elements: {int(mask_lottery.sum())}/{mask_lottery.numel()}")

In [ ]:
# Pathological Misuse: Arbitrary Layer Pruning vs Block Influence Guided Slicing
# Pruning high-BI layers collapses representation fidelity;
# Pruning low-BI layers preserves representation fidelity.
active_error = float(torch.norm(weights["x_in"] - weights["x_out_act"]))
redundant_error = float(torch.norm(weights["x_in"] - weights["x_out_red"]))
print(f"Active layer perturbation:    {active_error:.4f} (High information flow)")
print(f"Redundant layer perturbation: {redundant_error:.4f} (Identity mapping)")
assert redundant_error < 0.1 * active_error, "Redundant layer must have negligible influence compared to active layer!"
print("Pathology demonstrated: Blind layer dropping destroys representations, while ShortGPT BI pinpoints redundant identity layers.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Granularity vs. Speedup Tradeoff**: Unstructured magnitude pruning achieves highest parameter sparsity ($>80\%$) but requires custom sparse runtimes; structured row pruning physically resizes matrices, delivering immediate GEMM speedup on vanilla hardware.
2. **Zero-Shot Redundancy in LLMs**: ShortGPT Block Influence (BI) measures representation cosine distance, proving intermediate layers in deep transformers often act as near-identity transforms ($BI < 10^{-4}$) and can be removed without retraining.
3. **Winning Tickets**: Frankle & Carbin's Lottery Ticket Hypothesis demonstrates that dense overparameterization is essential during initial optimization, but sparse subnetworks isolated from initialization $W_0$ match full-model accuracy.

---
### Cross-Domain Generalisation Matrix
- **Quantization**: Structured pruning is isomorphic to block-wise bit allocation (HAWQ), dropping dimensions with zero second-order sensitivity.
- **Compilation**: Layer slicing transforms the graph IR directly, reducing transformer depth from $L \to L - k$ and eliminating memory allocation overhead.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_unstructured_structured_shortgpt_and_lottery_ticket.md](../../../markdown/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/01_unstructured_structured_shortgpt_and_lottery_ticket.md)
- **PyTorch 2.x Module:** [01_unstructured_structured_shortgpt_and_lottery_ticket.py](../../../pytorch2/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/01_unstructured_structured_shortgpt_and_lottery_ticket.py)
- **Functional JAX Module:** [01_unstructured_structured_shortgpt_and_lottery_ticket.py](../../../jax/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/01_unstructured_structured_shortgpt_and_lottery_ticket.py)
- **Native Rust Candle Track:** [01_unstructured_structured_shortgpt_and_lottery_ticket.rs](../../../candle/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/01_unstructured_structured_shortgpt_and_lottery_ticket.rs)
- **Hardware Kernel (CPU / WGSL):** [01_unstructured_structured_shortgpt_and_lottery_ticket.rs](../../../kernels/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/01_unstructured_structured_shortgpt_and_lottery_ticket.rs)
- **Golden SafeTensors:** [01_unstructured_structured_shortgpt_and_lottery_ticket.safetensors](../../../weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/01_unstructured_structured_shortgpt_and_lottery_ticket.safetensors)

---
**Curriculum Navigation:**  
&larr; Previous: [14. Quantisation / 06. Quantization Method Selection Matrix and Head to Head Benchmark](../../../notebooks/14_quantisation/05_kv_attention_optim_and_benchmarks/06_quantization_method_selection_matrix_and_head_to_head_benchmark.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 02. 2:4 Semi-Structured Sparsity and RigL Dynamic Sparse](02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.ipynb) &rarr;